# Phase 2: Tool-Calling Trajectory Generation

Generate training data by running the **GPT-5.4 teacher agent** on ~200 diverse tickers.

**Pipeline:**
1. Run `run_tool_calling_agent()` on each ticker with a random research focus
2. Save **raw trajectories** (full Responses API format) to `data/bbb/trajectories_raw.jsonl`
3. Convert to **Hermes/chat format** for SFT (truncate tool outputs, add `<think>` tags)
4. Quality filter and save to `data/bbb/trajectories_sft.jsonl`

**Rate limiting:** yfinance allows ~360 req/hr. With ~4-6 tool calls per ticker and 200 tickers,
we need 1-2s delays between tickers. Budget ~6-8 hours for the full run.

## Setup

In [1]:
import json
import os
import sys
import time
import random
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# Add nb/ to path so we can import bbb as a package
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_SCHEMAS, TOOL_FUNCTIONS
from bbb.agent import run_tool_calling_agent
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    FOCUS_AREAS,
    make_user_prompt,
    serialize_input_list,
    responses_to_hermes,
    filter_trajectory,
    trajectory_token_stats,
    count_tokens,
)

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(base_url="https://us.api.openai.com/v1")

MODEL = "gpt-5.4"

# Output paths
DATA_DIR = PROJECT_ROOT / "data" / "bbb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA_DIR / "trajectories_raw.jsonl"
SFT_PATH = DATA_DIR / "trajectories_sft.jsonl"

print(f"Tickers: {len(TICKERS)}")
print(f"Focus areas: {len(FOCUS_AREAS)}")
print(f"Raw output: {RAW_PATH}")
print(f"SFT output: {SFT_PATH}")

Tickers: 191
Focus areas: 8
Raw output: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_raw.jsonl
SFT output: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_sft.jsonl


## Configuration

Control the run size here. Set `MAX_TICKERS` to a small number for testing,
or `None` for the full ~200 ticker run.

In [2]:
# Set to a small number (e.g. 5) for testing, None for the full run
MAX_TICKERS = 5

# Delay between tickers (seconds) — yfinance rate limit is ~360 req/hr
DELAY_BETWEEN_TICKERS = 2.0

# Teacher model reasoning effort
REASONING_EFFORT = "medium"

# Shuffle tickers for diversity in partial runs
tickers = list(TICKERS)
random.shuffle(tickers)
if MAX_TICKERS is not None:
    tickers = tickers[:MAX_TICKERS]

print(f"Will generate trajectories for {len(tickers)} tickers")
print(f"Sample: {tickers[:10]}")

Will generate trajectories for 5 tickers
Sample: ['HUBS', 'BLK', 'COST', 'XEL', 'CME']


## Generate Raw Trajectories

Runs the teacher agent on each ticker and saves the full Responses API trajectory.
Results are written incrementally (one line per ticker) so partial runs are resumable.

In [3]:
# Load already-completed tickers (for resuming interrupted runs)
completed_tickers = set()
if RAW_PATH.exists():
    with open(RAW_PATH) as f:
        for line in f:
            record = json.loads(line)
            completed_tickers.add(record["ticker"])
    print(f"Resuming — {len(completed_tickers)} tickers already completed")

remaining = [t for t in tickers if t not in completed_tickers]
print(f"Remaining: {len(remaining)} tickers")

Remaining: 5 tickers


In [4]:
errors = []

for ticker in tqdm(remaining, desc="Generating trajectories"):
    focus = random.choice(FOCUS_AREAS)
    user_prompt = make_user_prompt(ticker, focus)

    try:
        result = run_tool_calling_agent(
            client=client,
            model=MODEL,
            user_prompt=user_prompt,
            system_prompt=SYSTEM_PROMPT,
            reasoning_effort=REASONING_EFFORT,
        )

        # Count tool calls from input list
        n_tool_calls = sum(
            1 for item in result["input"]
            if isinstance(item, dict) and item.get("type") == "function_call_output"
        )

        record = {
            "ticker": ticker,
            "focus": focus,
            "input": serialize_input_list(result["input"]),
            "output": serialize_input_list(result["output"]),
            "tools": TOOL_SCHEMAS,
            "reasoning_summaries": result["reasoning_summaries"],
            "usage": result["usage"],
        }

        # Append incrementally (resumable)
        with open(RAW_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")

        tqdm.write(
            f"  {ticker}: {n_tool_calls} tool calls, "
            f"{len(result['reasoning_summaries'])} reasoning steps, "
            f"memo: {len(result['output_text'] or '')} chars"
        )

    except Exception as e:
        errors.append((ticker, str(e)))
        tqdm.write(f"  {ticker}: ERROR — {e}")

    # Rate limit delay
    time.sleep(DELAY_BETWEEN_TICKERS)

print(f"\nDone. Errors: {len(errors)}/{len(remaining)}")
if errors:
    for t, e in errors:
        print(f"  {t}: {e}")

Generating trajectories:   0%|          | 0/5 [00:00<?, ?it/s]

  [1] Reasoning: **Gathering financial research on HubSpot**

I need to gather financials, price history, recommendations, and news for H...
  [1] Called get_financials(ticker='HUBS', statement_type='income', period='annual')
  [1] Called get_financials(ticker='HUBS', statement_type='income', period='quarterly')
  [1] Called get_financials(ticker='HUBS', statement_type='balance_sheet', period='annual')
  [1] Called get_financials(ticker='HUBS', statement_type='cashflow', period='annual')
  [1] Called get_price_history(ticker='HUBS', period='1y', interval='1wk')
  [1] Called get_recommendations(ticker='HUBS', months_back=12)
  [1] Called get_stock_news(ticker='HUBS')
  [2] Reasoning: **Synthesizing financial data**

I need to synthesize and calculate growth rates based on annual revenue figures: $1.731...
  [2] Reasoning: **Summarizing market performance**

I can discuss that the quarterly run-rate reached $847 million, with an operating in...
  [2] Reasoning: **Evaluating growth opport

## Inspect Raw Trajectories

Load the raw data and check sizes, tool usage patterns, and tool output lengths.

In [5]:
# Load all raw trajectories
raw_records = []
with open(RAW_PATH) as f:
    for line in f:
        raw_records.append(json.loads(line))

print(f"Total raw trajectories: {len(raw_records)}")

# Stats
tool_counts = []
tool_output_lengths = []
memo_lengths = []

for rec in raw_records:
    all_items = rec["input"] + rec["output"]

    # Tool calls
    calls = [it for it in all_items if isinstance(it, dict) and it.get("type") == "function_call"]
    tool_counts.append(len(calls))

    # Tool output sizes
    for it in all_items:
        if isinstance(it, dict) and it.get("type") == "function_call_output":
            tool_output_lengths.append(len(it["output"]))

    # Memo length
    for it in reversed(all_items):
        if isinstance(it, dict) and it.get("type") == "message":
            for c in it.get("content", []):
                if isinstance(c, dict) and c.get("text"):
                    memo_lengths.append(len(c["text"]))
                    break
            break

print(f"\nTool calls per trajectory: min={min(tool_counts)}, max={max(tool_counts)}, avg={sum(tool_counts)/len(tool_counts):.1f}")
print(f"Tool output chars: min={min(tool_output_lengths)}, max={max(tool_output_lengths)}, avg={sum(tool_output_lengths)/len(tool_output_lengths):.0f}")
print(f"Memo chars: min={min(memo_lengths)}, max={max(memo_lengths)}, avg={sum(memo_lengths)/len(memo_lengths):.0f}")

Total raw trajectories: 5

Tool calls per trajectory: min=6, max=14, avg=8.2
Tool output chars: min=2311, max=17132, avg=9974
Memo chars: min=7508, max=9704, avg=8301


## Convert to SFT Format (Hermes/Chat)

Convert from Responses API format → Hermes/chat format:
- `developer` → `system`
- Reasoning summaries → `<think>` tags in assistant content
- `function_call` items → `tool_calls` array on assistant messages
- `function_call_output` → `role: tool` messages
- Tool outputs **truncated** to ~600 tokens (zero-gradient masked tokens = pure overhead)
- Tool schemas converted from flat (Responses API) to nested (Chat Completions) format

In [6]:
# Convert and filter
sft_records = []
filtered_out = []

for rec in raw_records:
    # Quality check on raw trajectory
    passes, reason = filter_trajectory(rec)
    if not passes:
        filtered_out.append((rec["ticker"], reason))
        continue

    # Convert to Hermes format
    hermes = responses_to_hermes(rec, max_tool_tokens=600)
    if hermes is None:
        filtered_out.append((rec["ticker"], "conversion_failed"))
        continue

    hermes["ticker"] = rec["ticker"]
    hermes["focus"] = rec.get("focus", "")
    sft_records.append(hermes)

print(f"Converted: {len(sft_records)} trajectories")
print(f"Filtered out: {len(filtered_out)}")
if filtered_out:
    for t, r in filtered_out:
        print(f"  {t}: {r}")

Converted: 5 trajectories
Filtered out: 0


In [7]:
# Save SFT-ready data
with open(SFT_PATH, "w") as f:
    for rec in sft_records:
        f.write(json.dumps(rec) + "\n")

print(f"Saved {len(sft_records)} SFT trajectories to {SFT_PATH}")

Saved 5 SFT trajectories to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_sft.jsonl


## Inspect a Converted Trajectory

Check that the Hermes format looks correct before using it for SFT.

In [8]:
# Show the message flow of the first trajectory
if sft_records:
    sample = sft_records[0]
    print(f"Ticker: {sample['ticker']} — Focus: {sample['focus']}")
    print(f"Messages: {len(sample['messages'])}")
    print(f"Tools: {len(sample['tools'])}")
    print()

    for i, msg in enumerate(sample["messages"]):
        role = msg["role"].upper()
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", [])

        if role == "SYSTEM":
            print(f"[{i}] {role}: {content[:100]}...")
        elif role == "USER":
            print(f"[{i}] {role}: {content}")
        elif role == "ASSISTANT" and tool_calls:
            think = content[:120] + "..." if content and len(content) > 120 else content
            print(f"[{i}] {role} (with {len(tool_calls)} tool calls):")
            if think:
                print(f"     content: {think}")
            for tc in tool_calls:
                fn = tc["function"]
                print(f"     -> {fn['name']}({fn['arguments'][:80]})")
        elif role == "ASSISTANT":
            print(f"[{i}] {role} (final memo): {content[:150]}...")
        elif role == "TOOL":
            print(f"[{i}] {role} (call_id: {msg.get('tool_call_id', '?')[:20]}): {content[:100]}...")
        print()

Ticker: HUBS — Focus: growth potential and market expansion opportunities
Messages: 11
Tools: 4

[0] SYSTEM: You are an equity research analyst conducting comprehensive research on a given stock.

## Instructi...

[1] USER: Research HUBS focusing on growth potential and market expansion opportunities.

[2] ASSISTANT (with 7 tool calls):
     content: <think>
**Gathering financial research on HubSpot**

I need to gather financials, price history, recommendations, and ne...
     -> get_financials({"ticker":"HUBS","statement_type":"income","period":"annual"})
     -> get_financials({"ticker":"HUBS","statement_type":"income","period":"quarterly"})
     -> get_financials({"ticker":"HUBS","statement_type":"balance_sheet","period":"annual"})
     -> get_financials({"ticker":"HUBS","statement_type":"cashflow","period":"annual"})
     -> get_price_history({"ticker":"HUBS","period":"1y","interval":"1wk"})
     -> get_recommendations({"ticker":"HUBS","months_back":12})
     -> get_stock_news({"ti

## Dataset Statistics

Token counts per role, total trajectory lengths, and distribution analysis.
This tells us whether the data fits within `max_seq_length=8192` for SFT.

In [9]:
if sft_records:
    all_stats = [trajectory_token_stats(rec) for rec in sft_records]

    # Per-role averages
    for role in ["system", "user", "assistant", "tool", "total"]:
        vals = [s[role] for s in all_stats]
        print(f"{role:>10}: avg={sum(vals)/len(vals):.0f}  min={min(vals)}  max={max(vals)}")

    # How many exceed 8192?
    over_limit = sum(1 for s in all_stats if s["total"] > 8192)
    print(f"\nTrajectories > 8192 tokens: {over_limit}/{len(all_stats)}")
    if over_limit:
        print("(These may need further truncation or will be clipped during SFT)")

    system: avg=216  min=216  max=216
      user: avg=12  min=11  max=12
 assistant: avg=3555  min=2967  max=4589
      tool: avg=4963  min=3633  max=8460
     total: avg=8745  min=7131  max=13277

Trajectories > 8192 tokens: 1/5
(These may need further truncation or will be clipped during SFT)
